# Image Captioning

**Image captioning** generates a natural language description of an image. The `nlpconnect/vit-gpt2-image-captioning` model combines a **ViT image encoder** with a **GPT-2 text decoder** in an encoder-decoder architecture.

In this notebook we:
1. Load the captioning model
2. Generate captions for sample images
3. Display image + caption side by side
4. Explore how generation parameters affect caption length and style

## Setup

In [ ]:
!pip install transformers datasets sentence-transformers pillow -q

## Imports

In [ ]:
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}")

## 1. Load the Captioning Model

In [ ]:
MODEL_ID = "nlpconnect/vit-gpt2-image-captioning"
caption_model = VisionEncoderDecoderModel.from_pretrained(MODEL_ID).to(device)
feature_extractor = ViTImageProcessor.from_pretrained(MODEL_ID)
caption_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print(f"Model loaded: {MODEL_ID}")
print(f"Encoder: ViT | Decoder: GPT-2")

## 2. Generate Captions

In [ ]:
def generate_caption(image, max_length=30, num_beams=4):
    pixel_values = feature_extractor(images=image, return_tensors="pt").pixel_values.to(device)
    with torch.no_grad():
        output_ids = caption_model.generate(
            pixel_values,
            max_length=max_length,
            num_beams=num_beams,
            early_stopping=True,
        )
    return caption_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

def load_image(url):
    r = requests.get(url, timeout=10)
    return Image.open(BytesIO(r.content)).convert("RGB")

# Sample images
image_urls = {
    "Cat":         "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/320px-Cat_November_2010-1a.jpg",
    "Labrador":    "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg",
}

loaded, captions_generated = [], []
for name, url in image_urls.items():
    try:
        img = load_image(url)
        cap = generate_caption(img)
        loaded.append((name, img, cap))
        print(f"{name}: {cap}")
    except Exception as e:
        print(f"{name}: error ({e})")

## 3. Display Image + Caption Side by Side

In [ ]:
if loaded:
    fig, axes = plt.subplots(1, len(loaded), figsize=(6 * len(loaded), 5))
    if len(loaded) == 1:
        axes = [axes]
    for ax, (name, img, cap) in zip(axes, loaded):
        ax.imshow(img)
        ax.set_title(f"{name}\n\"{cap}\"", fontsize=9, wrap=True)
        ax.axis("off")
    plt.suptitle("Image Captioning with ViT-GPT2", fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("No images loaded — check network access.")

## 4. Effect of Generation Parameters

`num_beams` controls beam search width. `max_length` caps the caption length.

In [ ]:
if loaded:
    name, img, _ = loaded[0]
    print(f"Image: {name}\n")

    for num_beams in [1, 2, 4, 8]:
        cap = generate_caption(img, max_length=40, num_beams=num_beams)
        print(f"  num_beams={num_beams}: {cap}")

    print()
    for max_len in [10, 20, 40]:
        cap = generate_caption(img, max_length=max_len, num_beams=4)
        print(f"  max_length={max_len:2d}: {cap}")